# Volatility Modelling and Regime Classification Methods

**This is the corresponding code to the "Volatility Modelling and Regime Classification Methods" PDF. Please refer to the PDF for full explanations.**

Eloy Sentana Segui, 4th Year Bsc Mathematics and Computing, UC3M

Email: esentanasegui@gmail.com

LinkedIn: https://www.linkedin.com/in/eloysentanasegui

Github: https://github.com/eloysentana

## Introduction

In [ ]:
# Packages
import warnings

import arch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats
import statsmodels
import statsmodels.api as sm
import yfinance as yf
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Data

In [ ]:
# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Plot the closing price
fig = px.line(data, x=data.index, y="Close", title=f"^GSPC Closing Price")
fig.update_layout(
    xaxis_title="Date", 
    yaxis_title="Price",
    legend=dict(orientation="h", yanchor="bottom", y=-0.4, xanchor="center", x=0.5)
)
fig.write_image("closing_price.png", width=1700, height=600, scale=2)
fig.show()

### Simple Return vs Log Return

The simple return for a price series is defined as:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
- $R_t$: simple return at time $t$  
- $P_t$: price at time $t$  
- $P_{t-1}$: price at time $t-1$  

The logarithmic return (log return) for a price series is defined as:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

- $r_t$: log return at time $t$


In [ ]:
# Objective: Fit log-normal distributions to multiple international indices and plot histograms with fitted PDFs

# Select three international indices
international_tickers = ['^GSPC', '^KS11', '^N225', '^GDAXI']
index_names = {
    '^GSPC': 'S&P 500 (US)',
    '^KS11': 'KOSPI (South Korea)',
    '^N225': 'Nikkei 225 (Japan)',
    '^GDAXI': 'DAX (Germany)'
}

# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[index_names[t] for t in international_tickers]
)

row_col = [(1, 1), (1, 2), (2, 1), (2, 2)]

for idx, ticker in enumerate(international_tickers):
    row, col = row_col[idx]
    
    # Fetch data
    ticker_obj = yf.Ticker(ticker)
    hist = ticker_obj.history(start="2005-09-01", end="2025-10-01", interval="1d")
    prices = hist['Close'].dropna()
    
    # Use prices directly (no normalization)
    # Shift prices to ensure all values are positive if there are any issues
    prices_shifted = prices - prices.min() + 1  # Shift so minimum is 1
    
    # Fit log-normal distribution
    shape, loc, scale = stats.lognorm.fit(prices_shifted, floc=0)
    
    # Generate x values and PDF
    x = np.linspace(prices_shifted.min(), prices_shifted.max(), 1000)
    pdf = stats.lognorm.pdf(x, shape, loc=0, scale=scale)
    
    # Scale PDF to match histogram counts
    bin_width = (prices_shifted.max() - prices_shifted.min()) / 90
    pdf_scaled = pdf * len(prices_shifted) * bin_width
    
    # Add histogram with counts
    fig.add_trace(
        go.Histogram(
            x=prices_shifted,
            nbinsx=90,
            name=f'{index_names[ticker]} Histogram',
            marker_color='skyblue',
            opacity=0.6,
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Add fitted PDF (scaled to counts)
    fig.add_trace(
        go.Scatter(
            x=x,
            y=pdf_scaled,
            mode='lines',
            name=f'{index_names[ticker]} PDF',
            line=dict(color='red', width=2),
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Calculate and print statistics
    mean = stats.lognorm.mean(shape, loc=0, scale=scale)
    variance = stats.lognorm.var(shape, loc=0, scale=scale)
    print(f"\n{index_names[ticker]}:")
    print(f"  Fitted log-normal mean: {mean:.4f}")
    print(f"  Fitted log-normal variance: {variance:.4f} (std: {np.sqrt(variance):.4f})")

fig.write_image("lognormal_fits.png", width=1700, height=1200, scale=2)
fig.show()

In [ ]:
# Fetch price data and compute log returns
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Compute log returns
log_ret = np.log(data['Close']).diff().dropna()

# Get maximum return and minimum return with dates
max_rt = log_ret.max()
min_rt = log_ret.min()
max_rt_date = log_ret.idxmax().strftime('%Y-%m-%d')
min_rt_date = log_ret.idxmin().strftime('%Y-%m-%d')
print(f"Maximum (log) return: {max_rt*100:.2f}% on {max_rt_date}")
print(f"Minimum (log) return: {min_rt*100:.2f}% on {min_rt_date}")



# Plot the returns
mean_val = log_ret.mean()
fig = px.line(log_ret, x=log_ret.index, y=log_ret, title=f"^GSPC Log Return")
fig.add_hline(
    y=mean_val, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Mean: {mean_val:.4f}", 
    annotation_position="top left"
)
fig.update_layout(
    xaxis_title="Date", 
    yaxis_title="Log Return",
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5),
    showlegend=True
)
fig.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='lines',
        line=dict(color='red', dash='dash'),
        name=f"Mean: {mean_val:.4f}"
    )
)
fig.write_image("log_returns.png", width=1700, height=1200, scale=1)
fig.show()


In [ ]:
import plotly.graph_objects as go

ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate simple returns
simple_rets = data['Close'].pct_change().dropna()

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()

fig = go.Figure()

# Simple returns as red crosses
fig.add_trace(go.Scatter(
    x=simple_rets.index, y=simple_rets,
    mode='markers',
    marker=dict(symbol='x', color='red', size=7, opacity=0.4),
    name='Simple Returns'
))

# Log returns as blue circles
fig.add_trace(go.Scatter(
    x=log_rets.index, y=log_rets,
    mode='markers',
    marker=dict(symbol='circle', color='blue', size=5, opacity=0.4),
    name='Log Returns'
))

fig.update_layout(
    title="Scatter Plot of Simple vs Log Returns",
    xaxis_title="Date",
    yaxis_title="Return",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,
        xanchor="center",
        x=0.5,
        itemsizing='constant'
    )
)

fig.write_image("simple_vs_log_returns.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig.show()

### The distribution of the returns

In [ ]:
################################
# Input: simple and log return series
# output: comparative stats and comparative chart

# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

simple_rets = data['Close'].pct_change().dropna()
log_rets = np.log(data['Close']).diff().dropna()

# Create a DataFrame for comparative statistics
stats_df = pd.DataFrame({
    'Simple Return': [
        simple_rets.mean() * 100,
        simple_rets.std() * 100,
        simple_rets.skew(),
        simple_rets.kurtosis(),
        stats.jarque_bera(simple_rets)[0],
        stats.jarque_bera(simple_rets)[1]
    ],
    'Log Return': [
        log_rets.mean() * 100,
        log_rets.std() * 100,
        log_rets.skew(),
        log_rets.kurtosis(),
        stats.jarque_bera(log_rets)[0],
        stats.jarque_bera(log_rets)[1]
    ]
}, index=['Mean (%)', 'Std Dev (%)', 'Skew', 'Excess Kurtosis (Kurt - 3)', 'JB Stat', 'JB p-value'])

print('Comparative Statistics of Return Distributions')
print('-' * 60)
print(stats_df.round(4))
print()

# Normality interpretation
for label, ret in zip(['Simple', 'Log'], [simple_rets, log_rets]):
    jb_test = stats.jarque_bera(ret)
    if jb_test[1] < 0.05:
        print(f'{label} Return: Not normal (reject H0 at 5% significance level)')
    else:
        print(f'{label} Return: Normal (fail to reject H0 at 5% significance level)')
print()

# Plot both distributions on the same plot
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=simple_rets, 
    name=f'Simple Return (μ={simple_rets.mean()*100:.4f}%, σ={simple_rets.std()*100:.4f}%)', 
    opacity=1, 
    nbinsx=1000
))
fig.add_trace(go.Histogram(
    x=log_rets, 
    name=f'Log Return (μ={log_rets.mean()*100:.4f}%, σ={log_rets.std()*100:.4f}%)', 
    opacity=1, 
    nbinsx=1000
))
fig.update_layout(
    title='Simple vs Log Return Distribution',
    xaxis_title='Return',
    yaxis_title='Count',
    barmode='overlay',
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,
        xanchor="center",
        x=0.5
    )
)
fig.write_image("ret_distribution.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()



### Hypothesis testing for mean return and distribution normality

In [ ]:
# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()

alpha = 1e-2


# Null hypothesis for 'mean': the mean of the return series is zero
# Alternative hypothesis: the mean of the return series is not zero

t_stat, p = stats.ttest_1samp(log_rets, popmean=0, alternative='two-sided')
print("Mean test:")
print("p = {:g}".format(p))
if p < alpha:
    print("The null hypothesis can be rejected")
else:
    print("The null hypothesis cannot be rejected")


# Null hypothesis for 'normal': the return series is normally distributed
# Alternative hypothesis: the return series is not normally distributed

k2, p = stats.normaltest(log_rets)
print("Normality test:")
print("p = {:g}".format(p))
if p < alpha:
    print("The null hypothesis can be rejected")
else:
    print("The null hypothesis cannot be rejected")

## Volatility Modelling with GARCH

In [ ]:
# Get conditional volatility from the full model (already fitted in previous cells)

# First get the data from yfinance
import yfinance as yf
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go
from plotly.subplots import make_subplots
data = yf.Ticker("AAPL").history(period="5y")
log_rets = np.log(data['Close']).diff().dropna()

am_full = arch_model(log_rets, mean='constant', vol='Garch', p=1, q=1, dist='normal', rescale=True)
res_full = am_full.fit(disp='off')

cond_vol = res_full.conditional_volatility

# Calculate realized volatility with 10-day rolling window
realized_vol_10d = log_rets.rolling(window=10).std() * 100

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add GARCH conditional volatility
fig.add_trace(
    go.Scatter(
        x=cond_vol.index,
        y=cond_vol,
        name="GARCH Conditional Volatility",
        line=dict(color='blue', width=2)
    ),
    secondary_y=False
)

# Add realized volatility (10-day rolling)
fig.add_trace(
    go.Scatter(
        x=realized_vol_10d.index,
        y=realized_vol_10d,
        name="Realized Volatility (10-day)",
        line=dict(color='orange', width=1.5),
        opacity=0.7
    ),
    secondary_y=False
)

# Add log returns on secondary axis
fig.add_trace(
    go.Scatter(
        x=log_rets.index,
        y=log_rets * 100,
        name="Log Returns",
        line=dict(color='green', width=0.5),
        opacity=0.5
    ),
    secondary_y=True
)

# Update layout
fig.update_layout(
    title="GARCH Conditional Volatility, Realized Volatility (10-day) and Log Returns",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)

# Set axis titles
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Volatility (%)", secondary_y=False)
fig.update_yaxes(title_text="Log Returns (%)", secondary_y=True)

fig.write_image("garch_realized_returns.png", width=1700, height=1200, scale=2)
fig.show()

In [ ]:
# Let's plot the VIX vs actual volatility for S&P500 (21-day rolling std dev)
# First get the data from yfinance
sp = yf.Ticker('^GSPC').history(start="2005-09-01", end="2025-10-01", interval="1d")['Close']
vix = yf.Ticker('^VIX').history(start="2005-09-01", end="2025-10-01", interval="1d")['Close']

# Remove timezone information to allow proper merging
sp.index = sp.index.tz_localize(None)
vix.index = vix.index.tz_localize(None)

# Prepare the data
val_data = pd.DataFrame({
    'S&P500': sp,
    'VIX': vix
}).dropna()

val_data['Log Return'] = np.log(val_data['S&P500']).diff()
val_data['21-Day Volatility'] = val_data['Log Return'].rolling(window=21).std() * np.sqrt(252) * 100  # Annualized volatility in %

# Shift realized volatility by 21 days to align with forward-looking VIX
val_data['21-Day Volatility'] = val_data['21-Day Volatility'].shift(-21)

# Drop NaN values created by diff(), rolling(), and shift()
val_data = val_data.dropna()

# Plot VIX vs 21-Day Volatility
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=val_data.index,
    y=val_data['VIX'],
    mode='lines',
    name='VIX (Implied Volatility)',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=val_data.index,
    y=val_data['21-Day Volatility'],
    mode='lines',
    name='Realized Volatility (21-day forward)',
    line=dict(color='orange')
))

fig.update_layout(
    title='VIX vs Realized Volatility',
    xaxis_title="Date",
    yaxis_title="Volatility (%)",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="center",
        x=0.5
    ),
    hovermode="x unified"
)

# Also plot the all time means
fig.add_hline(
    y=val_data['VIX'].mean(),
    line_dash="dash",
    line_color="blue",
    annotation_text=f"Mean VIX: {val_data['VIX'].mean():.2f}",
    annotation_position="top left"
)

fig.add_hline(
    y=val_data['21-Day Volatility'].mean(),
    line_dash="dash",
    line_color="orange",
    annotation_text=f"Mean Realized Volatility: {val_data['21-Day Volatility'].mean():.2f}",
    annotation_position="bottom left"
)

fig.write_image("vix_vs_21day_volatility.png", width=1700, height=1200, scale=2)
fig.show()


# Plot a zoomed version between 2019-09-01 and 2020-08-01
fig.update_layout(
    title='VIX vs Realized Volatility (Zoomed)',
    xaxis_title="Date",
    yaxis_title="Volatility (%)",
    xaxis=dict(
        range=["2008-07-01", "2010-06-01"]
    ),
     legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="center",
        x=0.5
    ),
    hovermode="x unified"
)
fig.write_image("vix_vs_21day_volatility_zoomed.png", width=1700, height=1200, scale=2)
fig.show()

### Introduction

The basic idea is that the variance (volatility) at time $t$ depends on past squared returns (shocks) as well as past variances. This means that volatility tends to cluster around periods of high volatility.

But what do the acronym mean?

1. Generalized: Extends the simpler ARCH model by including past variances as explanatory variables, not just past shocks.
2. Autoregressive: The current volatility depends on its own past values (as in any [AR models](https://en.wikipedia.org/wiki/Autoregressive_model)).
3. Conditional: The variance is estimated with past information (i.e., it is not constant, but conditional on history).
4. Heteroskedasticity: The variance of the error term changes over time (we will return to this later).

The general GARCH(p, q) formula is:

$$
\sigma_t^2 = \omega + \sum_{i=1}^q \alpha_i \epsilon_{t-i}^2 + \sum_{j=1}^p \beta_j \sigma_{t-j}^2
$$

Where

$$
\begin{array}{ll}
\omega & \text{constant (baseline volatility)} \\ 
\sigma_t^2 & \text{conditional variance at time } t \\ 
\alpha_i, \beta_j & \text{parameters} \\ 
\epsilon_{t-i}^2 & \text{past squared shocks (errors)} \\ 
\sigma_{t-j}^2 & \text{past variances} \\ 
\end{array}
$$

$\omega$ ensures that the variance never collapses to zero

But... what is the shock $\epsilon_{t-i}^2$?
Mathematically, returns are modeled as:

$$
r_t = \mu_t + \varepsilon_t
$$


*Note that $\epsilon_t$ and $\varepsilon_t$ are used interchangeably (they are just variations of epsilon)*

That is, for every return, we try to predict it, but there is always some error (sometimes small, sometimes large) in our guess. That error is the shock.

And how do we get the expected return $\mu_t$? The answer is: it depends. Sometimes a constant is used; other times it is modeled with an [ARMA](https://didattica.unibocconi.it/mypage/dwload.php?nomefile=Lec_3_Autoregressive_Moving_Average_(ARMA)_Models_and_their_Practical_Applications20190212115606.pdf) process. In the case of a constant, it is obtained by "training" the model on past data. In some cases it is even set to 0 (in this case, we saw earlier that the mean was not significantly different from 0), because the main focus is volatility.

The shock is modeled as:

$$
\varepsilon_t = \sigma_t z_t
$$

where

$$
\begin{array}{ll}
\sigma_t & \text{conditional volatility at time } t \ (\text{changes over time, modeled by GARCH}) \\ 
z_t & \text{a random variable with mean 0 and variance 1 (often assumed i.i.d. normal)} \\ 
\end{array}
$$

And why do we square the shock in the GARCH formula? Because volatility must always be non-negative, and we care about its magnitude, not its direction.

With this information, we can better understand the intuition behind the GARCH model: if yesterday's volatility was high, today's volatility is also likely to be high.

Luckily for us, the most widespread version of GARCH is the GARCH(1,1) model ([see here](https://onlinelibrary.wiley.com/doi/10.1002/jae.800)):

$$
\sigma_t^2 = \omega + \alpha_1 \epsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2
$$

* Here, today's volatility depends on yesterday's squared shock and yesterday's volatility.

But there is an important property of GARCH: Covariance stationaryness

For this property, we need:

$$
\sum_{i=1}^q \alpha_i + \sum_{j=1}^p \beta_j < 1
$$

that ensures that the shocks eventually “die out,” and volatility returns to its long-run mean. If 
$
\sum_{i=1}^q \alpha_i + \sum_{j=1}^p \beta_j = 1
$
it is called IGARCH (integrated), where volatility persists at any horizon.
If 
$
\sum_{i=1}^q \alpha_i + \sum_{j=1}^p \beta_j > 1
$
 volatility explodes (i.e. it increases without bound, which is not the case for financial markets)

In [ ]:
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go
import yfinance as yf

# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns (scale by 100 for better model convergence)
log_rets = 100 * np.log(data['Close']).diff().dropna()

# === 1. Fit a simple GARCH(1,1) model ===
# Use default rescaling (rescale=True) which helps with optimization
am = arch_model(log_rets, vol='Garch', p=1, q=1, mean='Constant', dist='normal')
res = am.fit(disp="off", options={'maxiter': 1000})

# === 2. Get residuals and scale back to original units ===
# Divide by 100 to convert back 
errors = res.resid.dropna() / 100

# Split into 4 equal parts
splits = np.array_split(errors, 4)

# Collect variance + date ranges
results = []
split_boundaries = []
for i, s in enumerate(splits):
    start = s.index[0]
    end = s.index[-1]
    var = s.var()
    results.append((f"Period {i+1}", start, end, var))
    split_boundaries.append(end)  # for plotting

var_df = pd.DataFrame(results, columns=["Period", "Start", "End", "Variance"])

# === Plotly interactive plot of residuals ===
fig = go.Figure()

# Residuals line
fig.add_trace(go.Scatter(
    x=errors.index,
    y=errors.values,
    mode="lines",
    name="Model residuals",
    line=dict(color="blue")
))

# Mean line
mean_resid = errors.mean()
fig.add_hline(
    y=mean_resid,
    line=dict(color="green", dash="dot"),
    annotation_text=f"Mean = {mean_resid:.4f}",
    annotation_position="bottom right"
)

# Vertical split lines (using add_shape for datetime safety)
for boundary in split_boundaries[:-1]:
    fig.add_shape(
        type="line",
        x0=boundary,
        x1=boundary,
        y0=errors.min(),
        y1=errors.max(),
        line=dict(color="red", dash="dash"),
        xref="x",
        yref="y"
    )

# Layout
fig.update_layout(
    title="Model Residuals with Period Splits and Mean",
    xaxis_title="Date",
    yaxis_title="Residuals",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig.write_image("period_splits.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()


In [ ]:
print(var_df)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.stats import gaussian_kde

# === Create 2x2 subplot grid ===
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f"{var_df['Period'][i]} ({var_df['Start'][i].date()} to {var_df['End'][i].date()})<br>Mean = {splits[i].mean():.6f}  Var = {splits[i].var():.6f}"
    for i in range(4)
])

# === Loop through splits ===
for i, s in enumerate(splits):
    row = i // 2 + 1
    col = i % 2 + 1

    mean_val = s.mean()
    var_val = s.var()

    # Histogram (normalized to density for KDE overlay)
    fig.add_trace(
        go.Histogram(
            x=s.values,
            nbinsx=30,
            histnorm="probability density",
            marker_color="skyblue",
            showlegend=False
        ),
        row=row, col=col
    )

    # KDE curve (blue line)
    kde = gaussian_kde(s.values)
    x_range = np.linspace(s.min(), s.max(), 200)
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode="lines",
            line=dict(color="blue"),
            name="KDE",
            showlegend=(i == 0)  # only show legend once
        ),
        row=row, col=col
    )

    # Vertical line for mean (red dashed)
    fig.add_trace(
        go.Scatter(
            x=[mean_val, mean_val],
            y=[0, max(kde(x_range)) * 1.1],
            mode="lines",
            line=dict(color="red", dash="dash"),
            # name=f"Mean {i+1} = {mean_val:.4f}",   # each subplot shows its own mean
            showlegend=False
        ),
        row=row, col=col
    )

# === Layout ===
fig.update_layout(
    title=dict(
        text="Residuals Distribution by Period",
        x=0.5,  # center
        xanchor="center"
    ),
    template="plotly_white",
    autosize=True,
    margin=dict(l=40, r=40, t=80, b=40),  # tighter margins
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.05,
        xanchor="center",
        x=0.5
    )
)

# Show figure responsively (fills screen)
fig.write_image("residuals_distribution.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show(config={"responsive": True})


In [ ]:
from statsmodels.stats.diagnostic import het_arch

# === 4. ARCH test for heteroskedasticity ===
print("ARCH Test for Heteroskedasticity")
print("-" * len("ARCH Test for Heteroskedasticity"))
test_stat, p_value, _, _ = statsmodels.stats.diagnostic.het_arch(res.resid)
print("ARCH test statistic:", test_stat)
print("p-value:", p_value)
if p_value < 0.05:
    print("Evidence of heteroskedasticity: the variance of the errors changes over time.")
else:
    print("No significant heteroskedasticity detected.")


### Long-term variance under GARCH(1,1)

Let's now explore a concept that I find very interesting: What happens when we want to make a very far away prediction? Do we get the mean of all the variances filtered by our model? Or is there any other alternative?

In [ ]:
# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()


am = arch.univariate.arch_model(log_rets, x=None, 
                                mean='constant', lags=None, 
                                vol='Garch', p=1, o=0, q=1, 
                                dist='normal', hold_back=None, rescale=True)

volatility_model = am.fit()
volatility_model.summary()
# print(volatility_model.scale)

In [ ]:
volatility_model.params

In [ ]:
# Retrieve Model Parameters by name
omega = volatility_model.params['omega']
alpha = volatility_model.params['alpha[1]']
beta = volatility_model.params['beta[1]']

# long-term daily variance under GARCH
VL = omega / (1 - alpha - beta )
print(f'Long-term daily variance under GARCH (daily): {VL:.6f} / (annualized): {VL*252:.6f})') # we only multiply by 252!! not the sqrt bc its the variance

# long-term daily volatility under GARCH
sigma_L = np.sqrt(VL)
print(f'Long-term daily volatility under GARCH: {sigma_L:.2f} % / (annualized: {sigma_L*np.sqrt(252):.2f} %)')

# sample volatility estimate
sample_sigma = log_rets.std() * 100
print(f'Sample daily volatility estimate: {sample_sigma:.2f} % / (annualized: {sample_sigma*np.sqrt(252):.2f} %)')


### Conditional volatility

In [ ]:
garch_vol = volatility_model.conditional_volatility.round(5)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=garch_vol.index, y=garch_vol, name="GARCH Volatility"),
    secondary_y=False,
)

# Plot absolute value of the log returns, scaled to percentage to match garch_vol
fig.add_trace(
    go.Scatter(
        x=log_rets.index,
        y=log_rets.abs() * 100,  # scale to percentage
        name="|Log Return|",
        line=dict(color='orange', width=2),
        # fill='tozeroy',
        opacity=0.6
    ),
    secondary_y=False,  # use the same y-axis
)

fig.add_hline(
    y=float(sigma_L),
    line_dash="dash",
    line_color="green",
    annotation_text="Long-run volatility estimate: {:.2f}".format(float(sigma_L)),
    annotation_position="bottom left"
)
fig.add_hline(
    y=float(sample_sigma),
    line_dash="dot",
    line_color="red",
    annotation_text="Sample volatility: {:.2f}".format(float(sample_sigma)),
    annotation_position="top left"
)

fig.update_layout(
    title="GARCH(1,1) Volatility and Absolute Return",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)
fig.update_yaxes(title_text="Daily Volatility (%) % |Log Return| (%)", secondary_y=False)
fig.write_image("vol_vs_return_unzoom.png", width=1700, height=800, scale=1)

fig.show()


In [ ]:
# Filter data for the desired date range
start_date = "2020-02-16"
end_date = "2020-02-26"

garch_vol_zoom = garch_vol.loc[start_date:end_date]           # already in %
rt_zoom = log_rets.loc[start_date:end_date] * 100             # convert returns to %


fig = make_subplots(specs=[[{"secondary_y": True}]])

# Plot GARCH conditional volatility (daily %)
fig.add_trace(
    go.Scatter(
        x=garch_vol_zoom.index,
        y=garch_vol_zoom,
        name="GARCH Volatility",
        line=dict(color="blue", width=2)
    ),
    secondary_y=False,
)

# Plot absolute returns (daily %)
fig.add_trace(
    go.Scatter(
        x=rt_zoom.index,
        y=rt_zoom.abs(),
        name="|Return| %",
        line=dict(color='orange'),
        fill='tozeroy',
        opacity=0.3
    ),
    secondary_y=False,  # SAME y-axis scale
)

# Add reference lines (both already in %)
fig.add_hline(y=sigma_L, line_dash="dash", line_color="green", annotation_text="Long-run volatility estimate: " + str(sigma_L), annotation_position="bottom left")
fig.add_hline(y=sample_sigma, line_dash="dash", line_color="red", annotation_text="Sample volatility: " + str(sample_sigma), annotation_position="top left")

fig.update_layout(
    title="GARCH(1,1) Volatility and Absolute Return % (COVID-19 Crash Feb 16-26, 2020)",
    yaxis_title="%",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
)

fig.write_image("vol_vs_ret_superzoom.png", width=1700, height=800, scale=2)
fig.show()


In [ ]:
# Filter data for the desired date range
start_date = "2020-02-16"
end_date = "2020-06-05"

# GARCH volatility is already in % (since scale=100)
garch_vol_zoom = garch_vol.loc[start_date:end_date]

# Convert log returns to % to match scale
rt_zoom = log_rets.loc[start_date:end_date] * 100

fig = make_subplots(specs=[[{"secondary_y": True}]])

# GARCH volatility (daily %)
fig.add_trace(
    go.Scatter(
        x=garch_vol_zoom.index,
        y=garch_vol_zoom,
        name="GARCH Volatility",
        line=dict(color="blue", width=2)
    ),
    secondary_y=False,
)

# Absolute log returns (daily %)
fig.add_trace(
    go.Scatter(
        x=rt_zoom.index,
        y=rt_zoom.abs(),
        name="|Log Return| %",
        line=dict(color='orange'),
        fill='tozeroy',
        opacity=0.3
    ),
    secondary_y=False,  # SAME axis for both
)

# Reference lines: already in %
fig.add_hline(y=sigma_L, line_dash="dash", line_color="green", annotation_text="Long-run volatility estimate: " + str(sigma_L), annotation_position="bottom left")
fig.add_hline(y=sample_sigma, line_dash="dash", line_color="red", annotation_text="Sample volatility: " + str(sample_sigma), annotation_position="top left")

fig.update_layout(
    title="GARCH(1,1) Volatility and Absolute Return (Feb 16 - June 5, 2020)",
    yaxis_title="%",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
)

fig.write_image("vol_vs_ret_zoom.png", width=1700, height=800, scale=2)
fig.show()


In [ ]:
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go

# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()

# === 1. Fit the GARCH model ===
am = arch_model(log_rets, mean='constant', vol='Garch', p=1, q=1, dist='normal', rescale=True)
res = am.fit(disp="off")

# === 2. Conditional volatility from the model ===
cond_vol = res.conditional_volatility
print("Scale factor:", res.scale)

# === 3. Realized volatility (proxy) ===
window = 10  # rolling window length
realized_vol = log_rets.rolling(window).std()

# === 4. Make them comparable ===

# === 5. Align indices ===
df = pd.DataFrame({
    "Conditional Vol": cond_vol,
    "Realized Vol": realized_vol * res.scale
}).dropna()

# === 6. Plot with Plotly ===
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["Conditional Vol"],
    mode="lines",
    name="Conditional Volatility (GARCH)",
    line=dict(color="blue")
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["Realized Vol"],
    mode="lines",
    name=f"Realized Volatility ({window}-day rolling)",
    line=dict(color="orange"),
    opacity=0.7
))

fig.update_layout(
    title="Conditional vs Realized Volatility (rolling window = {} days)".format(window),
    xaxis_title="Date",
    yaxis_title="Volatility",
    legend=dict(x=0.01, y=0.99, bordercolor="Black", borderwidth=0.5),
    template="plotly_white"
)
fig.write_image("smoothed_ret_vs_vol.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()


## Future volatility forecasting with GARCH

We have seen how we can estimate the *past* volatility values *after* estimating its parameters. On this section however, we will focus on estimating *future* volatility, which is more useful for real life applications (where anticipating volatility levels helps hedge risks, or compute the level of exposure we have¡, like VaR modelling).

### Blind volatility prediction

In [ ]:
import arch
import pandas as pd
import numpy as np
from arch import arch_model
import yfinance as yf


# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()

train_ratio = 0.95

# Calculate split point
split_point = int(len(log_rets) * train_ratio)

# Split the data
train_period = log_rets.iloc[:split_point]
test_period = log_rets.iloc[split_point:]

print(f"Training data: {len(train_period)} observations ({train_period.index[0].strftime('%Y-%m-%d')} to {train_period.index[-1].strftime('%Y-%m-%d')})")
print(f"Test data: {len(test_period)} observations ({test_period.index[0].strftime('%Y-%m-%d')} to {test_period.index[-1].strftime('%Y-%m-%d')})")

# Train GARCH model on training data
print(f"\nTraining GARCH model on training data (from {train_period.index[0].strftime('%Y-%m-%d')} to {train_period.index[-1].strftime('%Y-%m-%d')})...")
am = arch.univariate.arch_model(train_period,
                                        mean='constant', lags=None,
                                        vol='Garch', p=1, o=0, q=1,
                                        dist='normal', rescale=True)

res = am.fit(disp='off')

print("GARCH parameters:\n", res.params)

cond_volatility = res.conditional_volatility

# Make predictions for the test period
print("Making predictions for test period...")
pred_volatility = res.forecast(horizon=len(test_period), reindex=False)
pred_volatility = np.sqrt(pred_volatility.variance.values[-1, :])

# Convert to pandas Series with proper dates
pred_volatility = pd.Series(pred_volatility, index=test_period.index)

# Calculate actual volatility for test period using rolling window


# For comparison, we'll use a simple approach: absolute returns
test_realized_vol = test_period.rolling(window=5).std() * 100
test_realized_vol = test_realized_vol.dropna()

# Calculate long-term volatility under GARCH(1,1)
params = res.params
omega = params['omega']
alpha = params['alpha[1]']
beta = params['beta[1]']
long_term_variance = omega / (1 - alpha - beta)
long_term_volatility = np.sqrt(long_term_variance)

# Calculate sample volatility
sample_volatility = log_rets.std()

prediction_results = {
    'train_returns': train_period,
    'test_returns': test_period,
    'train_fitted': res,
    'train_volatility': cond_volatility,
    'test_predictions': pred_volatility,
    'test_realized_vol': test_realized_vol,
    'split_point': split_point,
    'long_term_volatility': long_term_volatility,
    'sample_volatility': sample_volatility
}


In [ ]:
# Performance Evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_predictions(actual, predicted, garch_longterm_vol=None):
    """
    Evaluate prediction performance and compare with GARCH long-term volatility.
    """
    # Remove any NaN values for fair comparison
    valid_indices = ~(actual.isna() | predicted.isna())
    actual_clean = actual[valid_indices]
    predicted_clean = predicted[valid_indices]
    
    if len(actual_clean) == 0:
        print("No valid data points for evaluation")
        return {}
    
    mse = mean_squared_error(actual_clean, predicted_clean)
    mae = mean_absolute_error(actual_clean, predicted_clean)
    rmse = np.sqrt(mse)
    
    # Calculate additional metrics
    mape = np.mean(np.abs((actual_clean - predicted_clean) / actual_clean)) * 100
    correlation = np.corrcoef(actual_clean, predicted_clean)[0, 1]
    
    print("Prediction Performance Metrics:")
    print("-" * 40)
    print(f"Mean Squared Error (MSE): {mse:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
    print(f"Correlation: {correlation:.4f}")
    
    # Compare GARCH long-term volatility to test period average volatility
    if garch_longterm_vol is not None:
        test_avg_vol = actual_clean.mean()
        print("\nVolatility Comparison:")
        print("-" * 40)
        print(f"GARCH Long-term Volatility: {garch_longterm_vol:.2f}")
        print(f"Test Period Average Realized Volatility: {test_avg_vol:.2f}")
    
    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'correlation': correlation        
    }


# Align the predictions with realized volatility
aligned_predictions = prediction_results['test_predictions'].reindex(test_realized_vol.index)

# Evaluate performance and compare with GARCH long-term volatility
metrics = evaluate_predictions(
    test_realized_vol,
    aligned_predictions,
    garch_longterm_vol=prediction_results['long_term_volatility']
)

static_metrics = metrics

In [ ]:
def plot_garch_predictions(prediction_results, sigma_L=None, avg_test_vol=None, view="all", show_train_period=True):
    """
    Plot GARCH volatility predictions and forecasts.

    Parameters
    ----------
    prediction_results : dict
        Must contain keys: 'train_volatility', 'test_predictions', 'test_returns'
    sigma_L : float, optional
    avg_test_vol : float, optional
        Long-term volatility level to plot
    view : str, optional
        "all"      → show both subplots (default)
        "full"     → only complete time series with train/test split
        "zoom"     → only zoomed test + forecast
        "forecast" → only 30-day forecast with long-term volatility
    """

    if view == "all":
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=('Complete Time Series with Train/Test Split',
                           'Test Period Predictions and 30-Day Forecast'),
            vertical_spacing=0.1,
            row_heights=[0.6, 0.4]
        )
    else:
        fig = go.Figure()

    split_date = prediction_results['test_returns'].index[0]

    # ==== FULL SERIES VIEW ====
    if view in ["all", "full"]:
        if show_train_period:
            fig.add_trace(
                go.Scatter(
                    x=prediction_results['train_volatility'].index,
                    y=prediction_results['train_volatility'],
                    name="Training Volatility",
                    line=dict(color='blue', width=1),
                    opacity=0.7
                ),
                row=1 if view == "all" else None,
                col=1 if view == "all" else None
            )

        fig.add_trace(
            go.Scatter(
                x=prediction_results['test_predictions'].index,
                y=prediction_results['test_predictions'],
                name="Test Predictions",
                line=dict(color='red', width=2)
            ),
            row=1 if view == "all" else None,
            col=1 if view == "all" else None
        )



        if view == "full":
            fig.update_layout(title="GARCH Volatility - Full Series")
            fig.update_yaxes(title_text="Daily Volatility (%)")
            fig.update_xaxes(title_text="Date")

    # ==== ZOOM VIEW ====
    if view in ["all", "zoom"]:
        fig.add_trace(
            go.Scatter(
                x=prediction_results['test_predictions'].index,
                y=prediction_results['test_predictions'],
                name="Test Predictions (Zoomed)",
                line=dict(color='red', width=2),
                showlegend=(view == "zoom")
            ),
            row=2 if view == "all" else None,
            col=1 if view == "all" else None
        )

        if 'test_realized_vol' in globals():
            fig.add_trace(
                go.Scatter(
                    x=test_realized_vol.index,
                    y=test_realized_vol,
                    name="Realized Volatility",
                    line=dict(color='blue', width=1),
                    opacity=0.8
                ),
                row=2 if view == "all" else None,
                col=1 if view == "all" else None
            )

        if sigma_L is not None:
            fig.add_trace(
                go.Scatter(
                    x=[prediction_results['test_predictions'].index[0],
                       prediction_results['test_predictions'].index[-1]],
                    y=[sigma_L, sigma_L],
                    mode="lines",
                    line=dict(color="purple", width=2, dash="dot"),
                    name=f"Long-term Volatility = {sigma_L:.2f} %"
                ),
                row=2 if view == "all" else None,
                col=1 if view == "all" else None
            )
            
        if avg_test_vol is not None:
            fig.add_trace(
                go.Scatter(
                    x=[prediction_results['test_predictions'].index[0],
                       prediction_results['test_predictions'].index[-1]],
                    y=[avg_test_vol, avg_test_vol],
                    mode="lines",
                    line=dict(color="orange", width=2, dash="dot"),
                    name=f"Average Test Volatility = {avg_test_vol:.2f} %"
                ),
                row=2 if view == "all" else None,
                col=1 if view == "all" else None
            )

        if view == "zoom":
            fig.update_layout(title="GARCH Volatility - Test Period & Forecast")
            fig.update_yaxes(title_text="Daily Volatility (%)")
            fig.update_xaxes(title_text="Date")

    # Global layout
    fig.update_layout(
        height=600 if view != "all" else 800, 
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.1,
            xanchor="center",
            x=0.5
        )
    )
    return fig


In [ ]:
# Only the full series
myplot = plot_garch_predictions(prediction_results, view="full")
myplot.write_image("garch_full_series.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
myplot.show()


In [ ]:

# Only the zoomed test+forecast
myplot = plot_garch_predictions(prediction_results, sigma_L=prediction_results['long_term_volatility'], avg_test_vol=test_realized_vol.mean(), view="zoom")
myplot.write_image("garch_zoomed.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
myplot.show()


### Daily Retrain GARCH model


In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Suppress GARCH convergence warnings for cleaner output

def rolling_garch_forecast(returns, initial_window=0.90, forecast_steps=None, update_frequency=1):
    """
    Perform rolling window GARCH forecasting with model updates
    
    Parameters:
    returns: pandas Series of returns
    initial_window: fraction of data to use for initial training (default 0.8)
    forecast_steps: number of out-of-sample forecasts to make (default None → use all remaining steps)
    update_frequency: how often to re-estimate the model (1 = every day, 5 = every 5 days)
    
    Returns:
    Dictionary with rolling forecasts, actual volatility, and model parameters
    """
    
    
    # Calculate initial training window size
    initial_size = int(len(returns) * initial_window)
    total_obs = len(returns)
    
    # If forecast_steps not provided, use remaining observations
    if forecast_steps is None:
        forecast_steps = total_obs - initial_size
        print(f"forecast_steps set to {forecast_steps} (all remaining observations)")
    
    # Ensure we have enough data for forecasting
    if initial_size + forecast_steps > total_obs:
        forecast_steps = total_obs - initial_size
        print(f"Adjusted forecast_steps to {forecast_steps} due to data limitations")
    
    print(f"Initial training window: {initial_size} observations")
    print(f"Rolling forecasts: {forecast_steps} steps")
    print(f"Model update frequency: every {update_frequency} day(s)")
    
    # Initialize storage
    forecasts = []
    actual_returns = []
    actual_volatility = []
    forecast_dates = []
    model_params_history = []
    
    # Track when model was last updated
    last_update = -1
    current_model = None
    
    for i in range(forecast_steps):
        current_idx = initial_size + i
        
        # Determine training window end
        train_end = current_idx
        train_start = max(0, train_end - initial_size)  # Keep fixed window size
        
        # Get training data
        train_data = returns.iloc[train_start:train_end]
        
        # Update model if needed
        if current_model is None or (i - last_update) >= update_frequency:
            try:
                model = arch.univariate.arch_model(train_data, 
                                                   mean='constant', lags=None, 
                                                   vol='Garch', p=1, o=0, q=1, 
                                                   dist='normal', rescale=True)
                
                current_model = model.fit(disp='off', show_warning=False)
                last_update = i
                
                params = current_model.params.copy()
                params['update_step'] = i
                model_params_history.append(params)
                
            except Exception as e:
                print(f"Model fitting failed at step {i}: {e}")
                if current_model is None:
                    continue
        
        # Make 1-step ahead forecast
        try:
            forecast = current_model.forecast(horizon=1, reindex=False)
            vol_forecast = np.sqrt(forecast.variance.values[-1, 0])

            # print(f"Forecast at step {i}: {vol_forecast:.6f}")
            
            forecasts.append(vol_forecast)
            actual_returns.append(returns.iloc[current_idx])
            forecast_dates.append(returns.index[current_idx])
            
            realized_vol = abs(returns.iloc[current_idx]) * 100
            
            # print(f"Realized volatility at step {i}: {realized_vol:.6f}")

            actual_volatility.append(realized_vol)
            
        except Exception as e:
            print(f"Forecasting failed at step {i}: {e}")
            continue
        
        # if (i + 1) % 20 == 0:
        #     print(f"Completed {i + 1}/{forecast_steps} forecasts")
    
    forecasts = pd.Series(forecasts, index=forecast_dates, name='GARCH_Forecast')
    actual_volatility = pd.Series(actual_volatility, index=forecast_dates, name='Realized_Volatility')
    actual_returns = pd.Series(actual_returns, index=forecast_dates, name='Actual_Returns')
    
    print(f"\nCompleted rolling forecast with {len(forecasts)} successful predictions")
    
    return {
        'forecasts': forecasts,
        'actual_volatility': actual_volatility,
        'actual_returns': actual_returns,
        'model_params_history': model_params_history,
        'initial_window': initial_window,
        'forecast_steps': forecast_steps
    }

# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()

# Example execution
print("Starting rolling GARCH forecasting...")
rolling_results = rolling_garch_forecast(log_rets, initial_window=0.90, forecast_steps=None, update_frequency=1)



In [ ]:
# Visualize Rolling Forecasts

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

def plot_rolling_forecasts(rolling_results, show_last_n_days=None):
    forecasts = rolling_results['forecasts'].copy()
    actual_vol = rolling_results['actual_volatility'].copy()
    actual_returns = rolling_results['actual_returns'].copy()

    # ---- 1) Build a common index and reindex everything ----
    idx = forecasts.index.union(actual_vol.index).union(actual_returns.index)
    if show_last_n_days:
        idx = idx[-show_last_n_days:]
    forecasts = forecasts.reindex(idx)
    actual_vol = actual_vol.reindex(idx)
    actual_returns = actual_returns.reindex(idx)

    # ---- 2) Subplots with a shared x-axis ----
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=(
            'Rolling GARCH Volatility Forecasts vs Realized Volatility',
            'Forecast Errors Over Time',
            'Actual Returns'),
        row_heights=[0.5, 0.3, 0.3],
        vertical_spacing=0.03  # Reduce space between plots (default is 0.08)
    )

    # Top: forecasts vs actual
    fig.add_trace(go.Scatter(x=idx, y=forecasts, name="GARCH Forecasts",
                             line=dict(color='red', width=2),
                             mode='lines+markers', marker=dict(size=3)), row=1, col=1)

    fig.add_trace(go.Scatter(x=idx, y=actual_vol, name="Realized Volatility",
                             line=dict(color='blue', width=1),
                             mode='lines+markers', marker=dict(size=2), opacity=0.7), row=1, col=1)

    # Middle: errors (aligned by construction)
    forecast_errors = forecasts - actual_vol
    fig.add_trace(go.Scatter(x=idx, y=forecast_errors, name="Forecast Errors",
                             line=dict(color='green', width=1),
                             mode='lines+markers', marker=dict(size=2), showlegend=False), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

    # Bottom: returns
    fig.add_trace(go.Scatter(x=idx, y=actual_returns * 100, name="Daily Returns (%)",
                             line=dict(color='orange', width=1),
                             mode='lines', showlegend=False), row=3, col=1)

    # ---- 3) Force identical horizontal range (optional but explicit) ----
    xmin, xmax = idx.min(), idx.max()
    for r in (1, 2, 3):
        fig.update_xaxes(range=[xmin, xmax], row=r, col=1)

    # Labels & layout
    # Remove legend and add annotations for the first subplot
    # Place annotations near the middle of the plot for better visibility
    mid_idx = idx[len(idx)//2]
    quarter_idx = idx[len(idx)//4]
    fig.update_layout(
        title="Rolling GARCH Forecasting Results",
        height=700,
        showlegend=False,
        annotations=[
            dict(
                x=quarter_idx,
                y=forecasts.loc[quarter_idx],
                xref="x1", yref="y1",
                text="GARCH Forecasts",
                showarrow=True,
                arrowhead=1,
                ax=60, ay=-30,
                font=dict(color="red", size=15)
            ),
            dict(
                x=mid_idx,
                y=actual_vol.loc[mid_idx],
                xref="x1", yref="y1",
                text="Realized Volatility",
                showarrow=True,
                arrowhead=1,
                ax=60, ay=-30,
                font=dict(color="blue", size=15)
            )
        ]
    )
    fig.update_yaxes(title_text="Volatility (%)", row=1, col=1)
    fig.update_yaxes(title_text="Forecast Error", row=2, col=1)
    fig.update_yaxes(title_text="Return (%)", row=3, col=1)
    fig.update_xaxes(title_text="Date", row=3, col=1)

    return fig


# Create visualization
rolling_plot = plot_rolling_forecasts(rolling_results)
rolling_plot.write_image("rolling_forecasts_zoom.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

rolling_plot.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Model Parameter Evolution Analysis (with persistence, long-term volatility, and tabular stats)
def analyze_parameter_evolution(rolling_results):
    """
    Analyze how GARCH model parameters evolve over time,
    including persistence (alpha + beta) and long-term volatility.
    """
    
    params_history = rolling_results['model_params_history']
    
    if not params_history:
        print("No parameter history available")
        return
    
    # Convert to DataFrame
    params_df = pd.DataFrame(params_history)
    
    # Select key GARCH parameters
    garch_params = ['omega', 'alpha[1]', 'beta[1]']
    available_params = [p for p in garch_params if p in params_df.columns]
    
    # Add persistence if alpha and beta exist
    # Persistence formula: persistence = alpha[1] + beta[1]
    if 'alpha[1]' in params_df.columns and 'beta[1]' in params_df.columns:
        params_df['persistence'] = params_df['alpha[1]'] + params_df['beta[1]']
        available_params.append('persistence')
    
    # Add long-term volatility if omega, alpha, beta exist
    if all(p in params_df.columns for p in ['omega', 'alpha[1]', 'beta[1]']):
        denom = (1 - params_df['alpha[1]'] - params_df['beta[1]'])
        params_df['long_term_vol'] = np.sqrt(params_df['omega'] / denom.replace(0, np.nan))
        available_params.append('long_term_vol')
    
    if not available_params:
        print("GARCH parameters not found in history")
        return
    
    # Create parameter evolution plot
    fig = make_subplots(
        rows=len(available_params), cols=1,
        subplot_titles=[f'Parameter: {param}' for param in available_params],
        vertical_spacing=0.05  # reduced spacing between plots
    )
    
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    
    for i, param in enumerate(available_params):
        fig.add_trace(
            go.Scatter(
                x=params_df['update_step'],
                y=params_df[param],
                name=param,
                line=dict(color=colors[i % len(colors)], width=2),
                mode='lines',
            ),
            row=i+1, col=1
        )
        # Add mean line
        param_mean = params_df[param].mean()
        fig.add_hline(
            y=param_mean, 
            line_dash="dash", 
            line_color=colors[i % len(colors)], 
            opacity=0.5,
            row=i+1, col=1
        )

    # --- Plot percentage change of all metrics as a separate figure ---
    pct_change_df = pd.DataFrame()
    for i, param in enumerate(available_params):
        values = params_df[param].dropna()
        if len(values) == 0:
            continue
        start_val = values.iloc[0]
        pct_series = (params_df[param] / start_val) * 100 if start_val != 0 else np.nan
        pct_change_df[param] = pct_series

    pct_fig = go.Figure()
    for i, param in enumerate(pct_change_df.columns):
        pct_fig.add_trace(
            go.Scatter(
                x=params_df['update_step'],
                y=pct_change_df[param],
                name=param,
                line=dict(color=colors[i % len(colors)], width=2),
                mode='lines',
            )
        )
    pct_fig.update_layout(
        title="% Change of All Parameters (Start=100)",
        yaxis_title="% of Initial Value",
        xaxis_title="Update Step",
        legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5),
        height=500
    )
    
    fig.update_layout(
        title="GARCH Parameter Evolution During Rolling Forecast",
        height=int(400 * len(available_params) * 2 / 3),
        showlegend=False
    )
    
    # Update axis labels
    for i in range(len(available_params)):
        fig.update_yaxes(title_text="Value", row=i+1, col=1)
    
    # --- Print parameter statistics ---
    stats = {}
    for param in available_params:
        values = params_df[param].dropna()
        if len(values) == 0:
            continue
        first_val, last_val = values.iloc[0], values.iloc[-1]
        pct_change = ((last_val - first_val) / first_val * 100) if first_val != 0 else float("nan")
        
        stats[param] = {
            "Mean": values.mean(),
            "Std": values.std(),
            "Min": values.min(),
            "Max": values.max(),
            "% Change": pct_change
        }
    
    stats_df = pd.DataFrame(stats).T  # transpose to have params as rows
    print("Parameter Evolution Statistics:")
    print("=" * 70)
    print(stats_df.round(6).to_string())
    fig.write_image("garch_parameter_evolution.png", width=1700, height=900, scale=2)  # 170 mm ~ 1700 px at 300 dpi

    fig.show()

    pct_fig.write_image("combined_params_evo.png", width=1200, height=600, scale=2)
    pct_fig.show()
    
    return params_df, stats_df


# Example call:
params_evolution, stats_table = analyze_parameter_evolution(rolling_results)


### Train and Run approach

In [ ]:
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go

def garch_one_day_ahead(returns, split=0.9):
    """
    Run a GARCH(1,1) model on the first 'split' fraction of returns,
    then recursively compute 1-step-ahead variances for the rest.
    
    Parameters
    ----------
    returns : pd.Series
        Time series of returns
    split : float
        Fraction of sample used for initial estimation (default=0.9)
        
    Returns
    -------
    Dictionary with:
        - df: DataFrame with forecasts and realized shocks
        - res_train: fitted model on training sample
        - res_full: fitted model on full sample
        - split_idx: index that splits train/test
    """

    # Model 1: Trained daily, inferred daily
    # Model 2: Trained once on train dataset, inferred daily


    # === 1. Split sample ===
    #scale by 100 the returns
    returns = returns * 100

    n = len(returns)
    split_idx = int(n * split)
    train, test = returns.iloc[:split_idx], returns.iloc[split_idx:]
    # print(f"Training data: {len(train)} observations ({train.index[0].strftime('%Y-%m-%d')} to {train.index[-1].strftime('%Y-%m-%d')})")
    # print(f"Test data: {len(test)} observations ({test.index[0].strftime('%Y-%m-%d')} to {test.index[-1].strftime('%Y-%m-%d')})")
    
    # print(train.head(), test.head())
    # print("-"*40)
    # print(train.tail(), test.tail())

    # === 2. Fit GARCH(1,1) on training ===
    am = arch_model(train, mean="constant", vol="Garch", p=1, q=1, dist="normal", rescale=True)
    res_train = am.fit(disp="off")

    print("SCALE:", res_train.scale)

    mu = res_train.params["mu"]
    omega = res_train.params["omega"]
    alpha = res_train.params["alpha[1]"]
    beta  = res_train.params["beta[1]"]

    print("GARCH parameters (train):")
    print(res_train.params)
    
    # Last conditional variance from training
    sigma2_last = (res_train.conditional_volatility.iloc[-1])**2
    print("GARCH last conditional variance (train):", sigma2_last)
    
    # === 3. Recursive loop ===
    forecasts = []
    realized = []
    idx = []
    eps_prev = train.iloc[-1] - mu
    # print("Initial eps_prev:", eps_prev)
    sigma2_last = res_train.conditional_volatility.iloc[-1]**2
    # print("Initial sigma2_last:", sigma2_last)

    for t in range(len(test)):
        eps_realized = test.iloc[t] - mu
        sigma2_next = omega + alpha * eps_realized**2 + beta * sigma2_last
        # print(f"sigma2_next at t={t}: {sigma2_next}={omega} + {alpha}*{eps_realized**2} + {beta}*{sigma2_last}")
        sigma_next = np.sqrt(sigma2_next)
        forecasts.append(sigma_next)

        eps_realized = test.iloc[t] - mu
        realized.append(abs(eps_realized))
        idx.append(test.index[t])

        # Update recursion
        sigma2_last = sigma2_next


    # === 4. Fit full-sample model ===
    am_full = arch_model(returns, mean="constant", vol="Garch", p=1, q=1, dist="normal", rescale=True)
    res_full = am_full.fit(disp="off")
    
    # Pack results
    df = pd.DataFrame({
        "Forecast Volatility": forecasts,
        "Realized Shock": realized
    }, index=idx)
    
    return {
        "df": df,
        "res_train": res_train,
        "res_full": res_full,
        "split_idx": split_idx
    }


# First get the yahooo finance data
# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()

# === Example usage ===
results = garch_one_day_ahead(log_rets, split=0.3)
rolling_results = rolling_garch_forecast(log_rets, initial_window=0.3, forecast_steps=None, update_frequency=1) # daily retraining
# rolling_results = rolling_results


df = results["df"]
res_train = results["res_train"]
res_full = results["res_full"]
split_idx = results["split_idx"]


# === Plot 1: Forecast vs Realized Shock ===
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df["Forecast Volatility"],
    mode="lines", name="Train once model",
    line=dict(color="blue")
))
fig.add_trace(go.Scatter(
    x=df.index, y=rolling_results['forecasts'],
    mode="lines", name="Daily retrain model",
    line=dict(color="orange"), opacity=0.6
))
fig.update_layout(
    title="Daily trained vs Trained once volatility comparison",
    xaxis_title="Date",
    yaxis_title="Volatility%",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5)
)


# Show plots
fig.write_image("retrain_trainonce.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()


In [ ]:
# Plot the train once model vs absolute returns
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df["Forecast Volatility"],
    mode="lines", name="Train once model",
    line=dict(color="blue")
))
fig.add_trace(go.Scatter(
    x=df.index, y=(log_rets.loc[df.index].rolling(window=15).std() * 100) - res_train.params["mu"],
    mode="lines", name="Realized Volatility (15-day)",
    line=dict(color="red", width=1)
))
# Long-run volatility from full-sample model
params_full = res_full.params
omega_full = params_full["omega"]
alpha_full = params_full["alpha[1]"]
beta_full  = params_full["beta[1]"]
long_term_variance = omega_full / (1 - alpha_full - beta_full)
long_term_volatility = np.sqrt(long_term_variance)
fig.add_trace(go.Scatter(
    x=[df.index[0], df.index[-1]],
    y=[long_term_volatility, long_term_volatility],
    mode="lines",
    line=dict(color="purple", width=1, dash="dot"),
    name=f"Long-term Volatility = {long_term_volatility:.2f} %"
))
fig.update_layout(
    title="Train once GARCH Volatility vs (Smoothed) Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Volatility%",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5)
)
fig.write_image("trainonce_vs_absreturns.png", width=1700, height=900, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig.show()

# Print goodness-of-fit statistics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Calculate and print MSE for both methods (MSE, RMSE, MAE, MAPE, Correlation)

def calculate_gof_metrics(actual, predicted):
    """
    Calculate goodness-of-fit metrics between actual and predicted values.
    """
    # Remove NaN values for fair comparison
    valid_indices = ~(actual.isna() | predicted.isna())
    actual_clean = actual[valid_indices]
    predicted_clean = predicted[valid_indices]
    
    if len(actual_clean) == 0:
        print("No valid data points for evaluation")
        return {}
    
    mse = mean_squared_error(actual_clean, predicted_clean)
    mae = mean_absolute_error(actual_clean, predicted_clean)
    rmse = np.sqrt(mse)
    
    # Calculate additional metrics
    mape = np.mean(np.abs((actual_clean - predicted_clean) / actual_clean)) * 100
    correlation = np.corrcoef(actual_clean, predicted_clean)[0, 1]
    
    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'correlation': correlation        
    }

metrics_trainonce = calculate_gof_metrics(
    (log_rets.loc[df.index].rolling(window=15).std() * 100) - res_train.params["mu"], df["Forecast Volatility"]
)

print("Goodness-of-Fit Metrics for Train Once Model:")
print("-" * 40)
for key, value in metrics_trainonce.items():
    print(f"{key.upper()}: {value:.4f}")


In [ ]:
log_rets.describe()

# print(df["Forecast Volatility"].head(), df["Forecast Volatility"].tail())
# print(rolling_results['forecasts'].head(), rolling_results['forecasts'].tail())

# Plot the difference between the two forecast methods together with the plots of the forecasted volatility and the rolling result forecast
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df["Forecast Volatility"] - rolling_results['forecasts'],
    mode="lines", name="Difference",
    line=dict(color="red")
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["Forecast Volatility"],
    mode="lines", name="Trained-Once Model",
    line=dict(color="blue")
))
fig.add_trace(go.Scatter(
    x=rolling_results['forecasts'].index, y=rolling_results['forecasts'],
    mode="lines", name="Daily Trained Model",
    line=dict(color="orange"), opacity=0.6
))

fig.update_layout(
    title="Difference between Daily trained and Trained once volatility",
    xaxis_title="Date",
    yaxis_title="Volatility Difference",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5)
)

fig.write_image("daily_vs_trained_vs_diff.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()

fig.update_xaxes(range=["2019-09-01", "2020-08-01"])
fig.write_image("zoom_daily_vs_trained_vs_diff.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()


In [ ]:
# Now plot both methods against realized volatility
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.index, y=df["Forecast Volatility"],
    mode="lines", name="Train once model",
    line=dict(color="blue")
))
fig.add_trace(go.Scatter(
    x=rolling_results['forecasts'].index, y=rolling_results['forecasts'],
    mode="lines", name="Daily retrain model",
    line=dict(color="orange"), opacity=0.6
))
fig.add_trace(go.Scatter(
    x=df.index, y=df["Realized Shock"].rolling(window=10).std(),
    mode="lines", name="Realized Volatility (10-day rolling)",
    line=dict(color="green", width=0.8), opacity=1
))
# fig.add_trace(go.Scatter(
#     x=df.index, y=df["Realized Shock"],
#     mode="lines", name="Realized Volatility",
#     line=dict(color="black", width=0.5), opacity=0.7
# ))

fig.update_layout(
    title="GARCH One-day-ahead Volatility vs Realized Shock",
    # subtitle="Daily trained vs Trained once models",
    xaxis_title="Date",
    yaxis_title="Volatility%",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5)
)
# fig.write_image("garch_one_day_ahead_comparison.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at
# Show plots
# fig.write_image("garch_one_day_ahead_comparison.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig.show()

In [ ]:
# Plot the difference between the two methods and the realized volatility
# Calculate and print MSE for both methods
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Calculate rolling mean and align data
realized_rolling = df["Realized Shock"].rolling(window=5).mean()

# Align and drop NaN values for train-once model
aligned_data1 = pd.DataFrame({
    'realized': realized_rolling,
    'forecast': df["Forecast Volatility"]
}).dropna()

# Align and drop NaN values for rolling model
aligned_data2 = pd.DataFrame({
    'realized': realized_rolling,
    'forecast': rolling_results['forecasts']
}).dropna()

# Calculate metrics for train-once model
mse_trainonce = mean_squared_error(aligned_data1['realized'], aligned_data1['forecast'])
mae_trainonce = mean_absolute_error(aligned_data1['realized'], aligned_data1['forecast'])
rmse_trainonce = np.sqrt(mse_trainonce)
mape_trainonce = np.mean(np.abs((aligned_data1['realized'] - aligned_data1['forecast']) / aligned_data1['realized'])) * 100
corr_trainonce = np.corrcoef(aligned_data1['realized'], aligned_data1['forecast'])[0, 1]

# Calculate metrics for rolling model
mse_rolling = mean_squared_error(aligned_data2['realized'], aligned_data2['forecast'])
mae_rolling = mean_absolute_error(aligned_data2['realized'], aligned_data2['forecast'])
rmse_rolling = np.sqrt(mse_rolling)
mape_rolling = np.mean(np.abs((aligned_data2['realized'] - aligned_data2['forecast']) / aligned_data2['realized'])) * 100
corr_rolling = np.corrcoef(aligned_data2['realized'], aligned_data2['forecast'])[0, 1]

# Print comparison table
print("=" * 70)
print("Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<25} {'Train Once':<20} {'Daily Retrain':<20}")
print("-" * 70)
print(f"{'MSE':<25} {mse_trainonce:<20.6f} {mse_rolling:<20.6f}")
print(f"{'RMSE':<25} {rmse_trainonce:<20.6f} {rmse_rolling:<20.6f}")
print(f"{'MAE':<25} {mae_trainonce:<20.6f} {mae_rolling:<20.6f}")
print(f"{'MAPE (%)':<25} {mape_trainonce:<20.2f} {mape_rolling:<20.2f}")
print(f"{'Correlation':<25} {corr_trainonce:<20.6f} {corr_rolling:<20.6f}")
print("=" * 70)

# Calculate percentage improvement
print("\nPerformance Improvement (Daily Retrain vs Train Once):")
print("-" * 70)
print(f"MSE Improvement: {((mse_trainonce - mse_rolling) / mse_trainonce * 100):.2f}%")
print(f"MAE Improvement: {((mae_trainonce - mae_rolling) / mae_trainonce * 100):.2f}%")
print(f"MAPE Improvement: {((mape_trainonce - mape_rolling) / mape_trainonce * 100):.2f}%")
print(f"Correlation Improvement: {((corr_rolling - corr_trainonce) / corr_trainonce * 100):.2f}%")

In [ ]:

# train the model on the whole dataset
am_full = arch_model(log_rets * 100, mean="constant", vol="Garch", p=1, q=1, dist="normal", rescale=True)
res_full = am_full.fit(disp="off")

# val_data = pd.DataFrame()

# Now we plot the conditional volatility of the fully trained model, the train-once model, and the rolling retrain model
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=res_full.conditional_volatility.index,
    y=res_full.conditional_volatility,
    mode="lines",
    name="Conditional Volatility (Fully trained model)",
    line=dict(width=1.5)
))
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["Forecast Volatility"],
    mode="lines",
    name="Train Once Model",
    line=dict(width=1.5)
))
fig.add_trace(go.Scatter(
    x=rolling_results['forecasts'].index,
    y=rolling_results['forecasts'],
    mode="lines",
    name="Rolling Retrain Model",
    line=dict(width=1.5)
))
fig.update_layout(
    title="Conditional Volatility Comparison of GARCH Models",
    xaxis_title="Date",
    yaxis_title="Volatility%",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
)
fig.write_image("garch_conditional_vol_comparison.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig.show()


### Comparison with VIX


In [ ]:
# Data & processing
ticker = yf.Ticker("^VIX")
vix = ticker.history(start="2005-09-01", end="2025-10-01", interval="1d")
vix = vix.Close

ticker_sp = yf.Ticker("^GSPC")
data_sp = ticker_sp.history(start="2005-09-01", end="2025-10-01", interval="1d")
log_rets_sp = np.log(data_sp['Close']).diff().dropna()

# Fit GARCH on sp500 returns
am_sp = arch_model(log_rets_sp, mean="constant", vol="Garch", p=1, q=1, dist="normal", rescale=True)
res_sp = am_sp.fit(disp="off")

# Get conditional volatility (daily)
garch_vol = res_sp.conditional_volatility


# Clean the data by removing NaN values
cond_vol = res_sp.conditional_volatility * np.sqrt(252) # Annualize the volatility
vix_clean = vix.dropna()

print("Before date-only conversion:")
print(f"GARCH vol date range: {cond_vol.index[0]} to {cond_vol.index[-1]}")
print(f"VIX date range: {vix_clean.index[0]} to {vix_clean.index[-1]}")

# Fix timezone and time alignment by converting to date only
cond_vol.index = cond_vol.index.date
vix_clean.index = vix_clean.index.date

# Convert back to pandas datetime index (date only)
cond_vol.index = pd.to_datetime(cond_vol.index)
vix_clean.index = pd.to_datetime(vix_clean.index)

# print("After date-only conversion:")
# print(f"GARCH vol date range: {cond_vol.index[0]} to {cond_vol.index[-1]}")
# print(f"VIX date range: {vix_clean.index[0]} to {vix_clean.index[-1]}")

# Create DataFrame with proper alignment
val_data = pd.DataFrame({'VIX': vix_clean, 'GARCH': cond_vol}).dropna()

print(f"Final combined data shape: {val_data.shape}")
# print("Final data head:")
# print(val_data.head())
# print("...")
# print(val_data.tail())

In [ ]:
fig = px.line(val_data, line_shape='linear', title='VIX vs GARCH(1,1) Volatility')
fig.update_layout(
    legend=dict(orientation="h", yanchor="top", y=-0.1, xanchor="center", x=0.5, title=None),
    xaxis_title=None,
    yaxis_title="Annualized Volatility (%)",
    hovermode="x unified"
)
fig.write_image("vix_vs_garch_volatility.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig.show()

In [ ]:
fig = px.histogram(val_data.VIX - val_data.GARCH, title="Distribution of Diff = VIX - GARCH(1,1) Conditional Volatility", marginal="violin")
mean_diff = (val_data.VIX - val_data.GARCH).mean()
fig.add_vline(x=mean_diff, line_dash="dash", line_color="red", annotation_text=f"Mean: {mean_diff:.2f}", annotation_position="top left")
fig.update_layout(
    xaxis_title="VIX - GARCH(1,1)",
    yaxis_title="Count",
    showlegend=False
)
fig.write_image("vix_less_garch_distr.png", width=1700, height=600, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig.show()
(val_data.VIX - val_data.GARCH).describe()

## Regime Classifiers

### Hidden Markov Model

In [ ]:
def fitHMM(vol, n_states, random_state=0):
    # Initialize random state for reproducibility
    np.random.seed(random_state)

    train_vals = np.expand_dims(vol, 1)
    
    train_vals = np.reshape(train_vals,[len(vol),1])
    
    # fit Gaussian HMM to Q
    model = GaussianHMM(n_components=n_states, n_iter=1000).fit(train_vals)
     
    # classify each observation as state 0, 1 or 2
    hidden_states = model.predict(train_vals)
    post_prob = np.array(model.predict_proba(train_vals))
 
    # fit HMM parameters
    mus = np.squeeze(model.means_)
    sigmas = np.squeeze(np.sqrt(model.covars_))
    transmat = np.array(model.transmat_)
    # Print means, standard deviations, and transition matrix for inspection
    print("State means (mus):", mus)
    print("State standard deviations (sigmas):", sigmas)
    print("Transition matrix (transmat):\n", transmat)
    print("Model", model)
    
    relabeled_states = hidden_states
    return (relabeled_states, mus, sigmas, transmat, post_prob, model)

In [ ]:
def plot_model(dates, vol, post_prob, export_label):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=dates, y=vol, name="GARCH", mode='lines', line_shape='hv', yaxis = 'y1'))

    fig.add_trace(go.Scatter(x=dates, y=post_prob.iloc[:,0], name = 'Pr(Low Vol Regime)', mode='lines', line_shape='hv',
                             line=dict(width=0.5, color='green'), 
                             stackgroup='two', yaxis = 'y2'))

    fig.add_trace(go.Scatter(x=dates, y=post_prob.iloc[:,1], name = 'Pr(Medium Vol Regime)', mode='lines', line_shape='hv',
                             line=dict(width=0.5, color='orange'),
                             stackgroup='two', yaxis = 'y2'))

    fig.add_trace(go.Scatter(x=dates, y=post_prob.iloc[:,2], name = 'Pr(High Vol Regime)', mode='lines', line_shape='hv',
                             line=dict(width=0.5, color='red'),
                             stackgroup='two', yaxis = 'y2'))

    # Create axis objects
    fig.update_layout(
        title = ("Volatility Regime - " + str(export_label)),

        yaxis=dict(title="Volatility"),

        yaxis2=dict(title="Posterier Probability", overlaying="y1", side="right"),
        
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5)

    )

    # fig.write_html('Volatility Regime Classification - ' + str(export_label) + '.html') 
    fig.write_image(f'{export_label}.png', width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
    fig.show()

In [ ]:
# Get the GSPC data from yfinance
ticker_obj = yf.Ticker('^GSPC')
data = ticker_obj.history(start='2005-09-01', end='2025-10-01', interval='1d')

# Calculate log returns
log_rets = np.log(data['Close']).diff().dropna()

# Fit GARCH(1,1) model to get conditional volatility
am = arch_model(log_rets, mean="constant", vol="Garch", p=1, q=1, dist="normal", rescale=True)
res = am.fit(disp="off")

garch_vol = res.conditional_volatility

# hidden_states, mus, sigmas, transmat, post_prob, hmm_model = fitHMM((log_rets - log_rets.mean())*100, 3)
hidden_states, mus, sigmas, transmat, post_prob, hmm_model = fitHMM(garch_vol, 3)
dates = garch_vol.index

hmm_data = pd.DataFrame([dates, garch_vol, hidden_states], 
                        index = ["date", "garch_vol", "Most likely state"]).T

hmm_prob = pd.DataFrame(post_prob, columns = ['state_0', 'state_1', 'state_2'])
# hmm_prob is the probability of each state at each time point
hmm_data = pd.concat([hmm_data, hmm_prob], axis=1)


hmm_data.date = pd.to_datetime(hmm_data.date)
hmm_data = hmm_data.sort_values(by="date")

# hmm_data

In [ ]:
plot_model(hmm_data.date, hmm_data.garch_vol, hmm_prob, 'Hidden Markov Model')

### Markov Switching Autoregression Model 

In [ ]:
# Fit the model
np.random.seed(12345)
mod_hamilton = sm.tsa.MarkovAutoregression(log_rets-log_rets.mean(), k_regimes=3, order = 1, trend="n", switching_ar = True, switching_variance = True)
    
res_hamilton = mod_hamilton.fit()


res_hamilton.summary()


res_hamilton.params


In [ ]:
post_prob = res_hamilton.smoothed_marginal_probabilities
post_prob.columns = ['state_0', 'state_1', 'state_2']


plot_model(dates, garch_vol, post_prob, 'Markov Switching Autoregressive Model')


### Comparison

In [ ]:
hmm_log_prob = hmm_model.score(np.expand_dims(garch_vol.dropna(),1))
print('log-likelihood of HMM:', hmm_log_prob)
print('Transition Matrix of HMM:')
print(hmm_model.transmat_)

In [ ]:
msar_log_prob = mod_hamilton.loglike(res_hamilton.params)
trans_matrix = mod_hamilton.regime_transition_matrix(res_hamilton.params)
print('Log-likelihood of MSAR:', msar_log_prob)
print('Transition Matrix of MSAR:')
print(trans_matrix)

## References  
**By section:**  
1. Return distributional Assumptions  
> ARCH Model https://www.fsb.miamioh.edu/lij14/672_2014_s5.pdf  
> Heterogeneous Auroregressive Mean Model https://arch.readthedocs.io/en/latest/univariate/generated/arch.univariate.HARX.html#arch.univariate.HARX  
> Garch Forecasting Performance under Different Distribution Assumptions http://www-stat.wharton.upenn.edu/~steele/Courses/434/434Context/GARCH/Willhelmesson06.pdf  

2. Volatility Modelling
> Predicting volatility with heterogeneous autoregressive models https://www.sr-sv.com/predicting-volatility-with-heterogeneous-autoregressive-models/   


3. Hidden Markov Model
> Practical Time Series Analysis - code repo https://github.com/PracticalTimeSeriesAnalysis/BookRepo      
> HMMLearn https://hmmlearn.readthedocs.io/en/latest/
> Quantstrat HMM https://www.quantstart.com/articles/market-regime-detection-using-hidden-markov-models-in-qstrader/

4. Markov Switching Autoregressive Model
> ECB Volatility Regime https://www.ecb.europa.eu/pub/financial-stability/fsr/focus/2018/pdf/ecb~bcaaae16c3.fsrbox201805_03.pdf  
> Autoregressive conditional heteroskedasticity and changes in regime https://www.sciencedirect.com/science/article/abs/pii/0304407694900671    
> Markov-Switching - Kim, Nelson, and Startz (1998) Three-state Variance Switching http://www.chadfulton.com/topics/mar_kim_nelson_startz.html   
> Statsmodels Variance Switching Model https://www.statsmodels.org/dev/examples/notebooks/generated/markov_autoregression.html#Kim,-Nelson,-and-Startz-(1998)-Three-state-Variance-Switching  
> Statsmodels Markov Regression https://www.statsmodels.org/devel/examples/notebooks/generated/markov_regression.html   
